Cell 1: setup

In [ ]:
import os
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)            # run from the repo root so all data paths work
import h3
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Polygon

D, RAW = Path("data/processed"), Path("data/raw")
cells = pd.read_parquet(D / "cells_h3r7.parquet", columns=["cell", "range_rank"])
hexes = gpd.GeoDataFrame(cells, geometry=[Polygon([(lng, lat) for lat, lng in h3.cell_to_boundary(c)]) for c in cells["cell"]],
                         crs=4326).to_crs(3087)
hab = pd.read_parquet(D / "habitat_h3r7.parquet")
act = pd.read_parquet(D / "activity_h3r7.parquet")
cor = pd.read_parquet(D / "corridors_h3r7.parquet")
paths = gpd.read_parquet(D / "corridor_paths_h3r7.parquet").to_crs(3087)
g = hexes.merge(hab, on="cell").merge(act, on="cell").merge(cor.drop(columns="habitat_pct"), on="cell")

def basemap(ax, title):
    hexes.plot(ax=ax, color="#eeeeee", linewidth=0)
    ax.set_title(title, fontsize=11)
    ax.set_axis_off()

Cell 2: likely habitat, with the tracking data

In [ ]:
from bearcast.bears.habitat import clean_gps, clean_radio
radio = clean_radio(pd.read_parquet(RAW / "fwc_telemetry_combined.parquet",
                                    columns=["Source", "Bear_ID", "Date", "Loc_Type", "Latitude", "Longitude"]))
gps = clean_gps(pd.read_parquet(RAW / "fwc_telemetry_camp_blanding.parquet",
                                columns=["Bear", "GMT_DATE", "LATITUDE", "LONGITUDE", "DOP"]))
fixes = pd.concat([radio, gps], ignore_index=True)
fx = gpd.GeoDataFrame(fixes, geometry=gpd.points_from_xy(fixes["lon"], fixes["lat"]), crs=4326).to_crs(3087)
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
basemap(axes[0], "Likely bear habitat (percentile, 1 = best)")
g.plot(ax=axes[0], column="habitat_pct", cmap="Greens", linewidth=0, legend=True, legend_kwds={"shrink": 0.6})
basemap(axes[1], f"Tracking data used to train it ({fixes['bear'].nunique()} bears, {len(fixes):,} daily locations)")
g.plot(ax=axes[1], column="habitat_pct", cmap="Greens", linewidth=0, alpha=0.5)
fx.plot(ax=axes[1], column="group", markersize=0.3, legend=True, legend_kwds={"loc": "lower left", "fontsize": 8})
plt.tight_layout()

Cell 3: recent activity

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
basemap(ax, "Recent bear activity: reports in the last 90 days (smoothed)")
colors = {"low": "#fee6ce", "moderate": "#fdae6b", "high": "#e6550d"}
for lvl, col in colors.items():
    g[g["activity_level"] == lvl].plot(ax=ax, color=col, linewidth=0)
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in colors.values()], labels=list(colors), loc="lower left")
plt.tight_layout()

Cell 4: corridors between populations

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
basemap(ax, "Population cores (green), corridor bands (darker = closer to the best route) and best routes")
g[g["corridor_value"] <= 0.15].plot(ax=ax, column="corridor_value", cmap="YlOrBr_r", linewidth=0, vmin=0, vmax=0.15)
g[g["core"] != ""].plot(ax=ax, color="#1b7837", linewidth=0)
paths.plot(ax=ax, color="black", linewidth=1)
for name, pt in g[g["core"] != ""].dissolve("core").representative_point().items():
    ax.annotate(name, (pt.x, pt.y), fontsize=8, ha="center", color="white",
                bbox={"boxstyle": "round", "fc": "#1b7837", "ec": "none", "alpha": 0.8})
print(paths.drop(columns="geometry").round(1).to_string(index=False))
plt.tight_layout()

Cell 5: road-crossing risk vs real roadkill

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
basemap(ax, "Road-crossing risk (top 5% of road hexagons) and roadkill 2010-2025")
road = g[g["crossing_risk"] > 0]
road[road["crossing_risk"] >= road["crossing_risk"].quantile(0.95)].plot(ax=ax, color="#d6604d", linewidth=0)
rk = g[g["roadkill_2010_2025"] > 0]
rk.assign(geometry=rk.geometry.centroid).plot(ax=ax, color="black", markersize=rk["roadkill_2010_2025"].clip(upper=20))
plt.tight_layout()

Cell 6: why Wekiva’s top-rated habitat got almost no use

In [ ]:
from bearcast.bears import habitat as H
from bearcast.features.landscape import load_cells
cg = load_cells(D / "cells_h3r7.parquet")
static = pd.read_parquet(D / "landscape_h3r7.parquet").merge(pd.read_parquet(D / "landcover_h3r7.parquet"), on="cell")
d, feats = H.add_features(H.build_design(fixes, set(cg["cell"])), static, H.era_housing_table(cg))
train, test = d[d["group"] != "wekiva"], d[d["group"] == "wekiva"].copy()
test["score"] = H.fit(train, feats).predict(test[feats].to_numpy(np.float32))
av = test[test["used"] == 0]
top = av[av["score"] >= av["score"].quantile(0.9)]
used_cells = test.loc[test["used"] == 1, "cell"].unique()
fig, ax = plt.subplots(figsize=(8, 8))
area = hexes[hexes["cell"].isin(av["cell"])]
area.plot(ax=ax, color="#eeeeee", linewidth=0)
hexes[hexes["cell"].isin(top["cell"])].plot(ax=ax, color="#d6604d", linewidth=0)
hexes[hexes["cell"].isin(used_cells)].plot(ax=ax, color="#2166ac", linewidth=0, alpha=0.7)
ax.set_title("Wekiva (held out): grey = available, red = model's top 10%, blue = hexagons the bears used")
ax.set_axis_off()
key = ["dist_bigcons_km", "cons_share", "lc_natural", "lc_forested_wetland", "lc_water", "hu_den_era", "dist_major_road_km"]
print(pd.DataFrame({"model top 10%": top[key].mean(), "used by bears": test.loc[test["used"] == 1, key].mean(),
                    "all available": av[key].mean()}).round(2).to_string())
plt.tight_layout()